<a href="https://colab.research.google.com/github/lack-of-sleep/2609Study/blob/main/18.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Section 18**
n-gram 언어 모델 기본형 만들기

**N-gram**

**I am a student**

1-gram : I, am, a, student

2-gram : I am, am a, a student

3-gram : I am a, am a student

In [2]:
def ngram(text, n):
  return zip(*[text[i:] for i in range(n)])

#입력받은 text에서 n개의 음절/단어로 묶음


In [3]:
print([i for i in ngram("오늘 날씨는 비", 3)])

[('오', '늘', ' '), ('늘', ' ', '날'), (' ', '날', '씨'), ('날', '씨', '는'), ('씨', '는', ' '), ('는', ' ', '비')]


In [4]:
sentence_list = "It is raining today".split()
print(sentence_list)
print([i for i in ngram(sentence_list, 3)])


['It', 'is', 'raining', 'today']
[('It', 'is', 'raining'), ('is', 'raining', 'today')]


In [5]:
!pip install konlpy
from konlpy.tag import Okt

okt = Okt()

sentence_list = [w for w in Okt().morphs("오늘 날씨는 비")]
#Okt를 이용해 텍스트를 형태소 단위로 토큰화
print(sentence_list)

print(["".join(li) for li in ngram(sentence_list,3)])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 34.4 MB/s eta 0:00:00
['오늘', '날씨', '는', '비']
['오늘날씨는', '날씨는비']


In [7]:
from collections import defaultdict
#기본값 자동생성 딕셔너리 사용

def train(text,n):
  model = defaultdict(lambda: defaultdict(int))
  for w in ngram(text,n):
    model[w[:-1]][w[-1]] += 1
  return model
  #w[:-1] = 처음부터 마지막 전까지 = 컨텍스트
  #w[-1] = 마지막 = 다음단어
  #해당 조합의 빈도수를 +1 하여 카운트

train(sentence_list,3)
#오늘 날씨 다음에 는이 1번
#날씨 는 다음에 비가 1번

defaultdict(<function __main__.train.<locals>.<lambda>()>,
            {('오늘', '날씨'): defaultdict(int, {'는': 1}),
             ('날씨', '는'): defaultdict(int, {'비': 1})})

In [8]:
def predict(model, n, prefix):
  next_words = []
  for w in ngram(prefix, n-1):
    for next_word, count in model[w].items():
      next_words.append((next_word, count))
  return sorted(next_words, key=lambda x: x[1], reverse=True)


model = train(sentence_list,3)
predict(model, 3, sentence_list[:-1])

[('는', 1), ('비', 1)]

In [12]:
from collections import defaultdict
from konlpy.tag import Okt

class NgramLM:
  def __init__(self):
    self.model = defaultdict(lambda: defaultdict(int))

  def tokenize(self, text, word=True):
    if word:
      self._tokenizer = Okt().morphs
      return [w for w in Okt().morphs(text)]
    else:
      return text

  def ngram(self, text, n):
    return zip(*[text[i:] for i in range(n)])

  def train(self, text, n, word=True):
    if word:
      self.trained = "word"
      text = self.tokenize(text,word)
    else:
      self.trained = "char"

    for w in self.ngram(text,n):
      self.model[w[:-1]][w[-1]] += 1

  def predict(self, prefix, n, word=True):
    if self.trained == "word" and word:
      prefix = self.tokenize(prefix,word)
    elif self.trained == "char" and word:
      return "err:tokenizer of train/predict is different"

    next_words = []
    for w in self.ngram(prefix, n-1):
      for next_word, count in self.model[w].items():
        next_words.append((next_word, count))
    return sorted(next_words, key=lambda x: x[1], reverse=True)[0][0]

ngram_lm = NgramLM()

ngram_lm.train("나는 고양이. 나는 강아지. 나는 사람. 나는 고양이.",3,True)

ngram_lm.predict("나는",3,True)


'고양이'

In [16]:
class Clock:
  def __init__(self):
    self.hour=0
    self.minute=0

  def set_time(self, hour, minute):
    if(0<=hour<12) and (0<=minute<60):
      self.hour = hour
      self.minute = minute
      print(f"시간을 {self.hour}:{self.minute:02} (으)로 설정했습니다.")

    else: print("시간 설정이 잘못되었습니다.")

  def show_time(self):
    print(f"지금 시간은 {self.hour}:{self.minute:02} 입니다.")

clock = Clock()
clock.set_time(11,59)
clock.show_time()

시간을 11:59 (으)로 설정했습니다.
지금 시간은 11:59 입니다.
